1. Parâmetros do ambiente 

In [0]:
catalog = "projeto_cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/bronze/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

2. Criação do Catalog, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e volume prontos.")

3. Validação da Landing Zone

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

4. Mapeamento dos caminhos no Volume e leitura pura

In [0]:
from pyspark.sql.functions import current_timestamp

path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=True)
df_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=True)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=True)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=True)

5. Gravação com adição do timestamp no momento da escrita

In [0]:
df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("Append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("Append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("Append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("Append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("Append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")

6. Visualização prévia das tabelas

In [0]:
tables = [
    "tb_credits_and_tags",
    "tb_movies_financials",
    "tb_movies_info",
    "tb_movies_metrics",
    "tb_movies_reviews"
]

for table in tables:
    print(f"\n===== {table} =====")
    display(
        spark.table(f"{catalog}.bronze.{table}").limit(5)
    )

7. Visualizar se o timestamp realmente foi adicionado as tabelas

In [0]:
for table in tables:
    df = spark.table(f"{catalog}.bronze.{table}")

    print(f"\n===== {table} =====")
    print(f"Linhas: {df.count()}")
    print(f"Colunas: {len(df.columns)}")
    
    df.printSchema()

8. Adicionei para comparar com a quantidade de linhas originais do landing para ter certeza que o Append nn duplicou as linhas

In [0]:
tables = [
    "tb_credits_and_tags",
    "tb_movies_financials",
    "tb_movies_info",
    "tb_movies_metrics",
    "tb_movies_reviews"
]

for table in tables:
    df_bronze = spark.table(f"{catalog}.bronze.{table}")
    
    print(f"{table}: {df_bronze.count()} linhas")

In [0]:
landing_files = {
    "tb_credits_and_tags": "credits_and_tags_IMDB_TMDB.csv",
    "tb_movies_financials": "movies_financials_IMDB_TMDB.csv",
    "tb_movies_info": "movies_info_TMDB_IMDB.csv",
    "tb_movies_metrics": "movies_metrics_IMDB_TMDB.csv",
    "tb_movies_reviews": "movies_reviews.csv"
}

for table, filename in landing_files.items():
    path = f"{landing_path}/{filename}"
    
    df_landing = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )
    
    print(f"{table}: {df_landing.count()} linhas na Landing")

9. Importanto o tempo como widgets para usar na API

In [0]:
from datetime import datetime, timedelta
import requests

data_fim_default = datetime.now()
data_inicio_default = data_fim_default - timedelta(days=7)

dbutils.widgets.text("data_inicio", "01-01-2016", "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_default.strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

In [0]:
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Data início: '{data_inicio}'")
print(f"Data fim: '{data_fim}'")

10. Montar a URL da API

In [0]:
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

print(f"Consultando API:\n{url}")

11. Fazer a requisição

In [0]:
response = requests.get(url)
dados_json = response.json().get("value", [])

df_cotacao_raw = spark.createDataFrame(dados_json)

12. Salvando a tabela de cotação no catalogo bronze

In [0]:
(
    df_cotacao_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
)

print(f"[OK] Tabela {bronze_schema}.tb_cotacao_dolar salva com sucesso. Total de registros: {df_cotacao_raw.count()}")

12. Verificar a tabela

In [0]:
display(
    spark.table(
        f"{catalog}.bronze.tb_cotacao_dolar"
    )
)

13. Verificar o schema

In [0]:
spark.table(
    f"{catalog}.bronze.tb_cotacao_dolar"
).printSchema()